[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/math/penrose/penrose.ipynb)

# Penrose Tilings

This example constructs a non-periodic Penrose rhombus tiling and its local phason flips by slicing the five-dimensional integer lattice with a two-dimensional plane at a five-fold slope. In the cut-and-project model, each square face of the five-dimensional lattice whose dual three-cube intersects the tilted slice projects onto a thick or thin rhombus on the slice. Shifting the slice perpendicularly without changing its slope alters which dual cubes meet the slice, rearranging the tiling by local three-tile hexagon flips without translating the pattern as a whole.

Four reflections across the mirrors between consecutive lattice axes compose into an order-five rotor `turn`, and averaging the five powers of `turn >> Point` with cosine weights isolates the harmonic projections `cut_projection` onto the slice and `phason_projection` onto the perpendicular phason plane, making the slice the join `cut_projection(axes[0]) & cut_projection(axes[1])`. Cramer's rule in joins alone builds each dual cube's reciprocal reading planes `(second & third & cut) / (first & second & third & cut)`, testing all three dual-cube offsets by metric-free pairings `frames & offset`, while comparing `(cut_projection(first) & cut_projection(second)).norm()` against the perpendicular three-join shadow norm shows that thick rhombi outnumber thin ones by the golden ratio.

In the pentagrid notation of [de Bruijn (1981)](https://doi.org/10.1016/1385-7258(81)90016-0) and in $5 \times 5$ orthogonal-matrix notation, five planar unit star vectors $\mathbf e_j = (\cos\tfrac{2\pi j}{5}, \sin\tfrac{2\pi j}{5})$ and perpendicular star vectors $\mathbf e_j^\perp = (\cos\tfrac{4\pi j}{5}, \sin\tfrac{4\pi j}{5})$ with shifts $\sum_{j=0}^4 \gamma_j = 0$ locate each rhombus at the intersection of two grid lines $\mathbf x \cdot \mathbf e_r + \gamma_r = k_r$ and $\mathbf x \cdot \mathbf e_s + \gamma_s = k_s$, assign the remaining integer coordinates by ceiling step functions $k_j = \lceil \mathbf x \cdot \mathbf e_j + \gamma_j \rceil$, and evaluate $3 \times 3$ subdeterminants of the $5 \times 5$ cyclic Fourier matrix to test acceptance in perpendicular space.

In [ ]:
# In Colab, install numga and its examples from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
import itertools
from collections.abc import Iterator

import numpy as np

from numga import Algebra, NumpyContext, stack
from examples.util.animation import save_figure, show_states
from examples.math.penrose import render, scenarios

np.set_printoptions(precision=3, suppress=True)

ga = Algebra("a+b+c+d+e+")
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                                   # Scalar+
# Mirrors and reading planes that pair with points and offsets.
Plane = ga.gatype.vector()                                                    # Plane+
# Lattice points and axis directions are intrinsic antivectors; joins of two span oriented areas.
Point = ga.gatype.antivector()                                                # Point-
Area = ga.gatype.antibivector()                                               # Area-
Projection = ga.gatype((Point, Point))                                        # Point- <- Point-

# Coordinate mirrors across the five axes and their dual axis directions.
mirrors = mv.basis()                                                          # [5] Plane+
axes = mirrors.dual()                                                         # [5] Point-
# The ten axis pairs spanning lattice square faces, and the three remaining axes of each dual cube.
face_pairs = list(itertools.combinations(range(5), 2))
side_indices = np.array(face_pairs)                                           # [10, 2]
other_indices = np.array([[index for index in range(5) if index not in pair] for pair in face_pairs])  # [10, 3]

## Five-fold turn and harmonic projections

Each reflection across the mirror bisecting two consecutive coordinate axes swaps them, so four reflections in sequence cycle all five axes in an order-five rotation. Because cosines of multiples of a fifth of a turn are orthogonal over five steps, averaging the five powers of `turn >> Point` with cosine weights isolates the eigenplane that turns by one fifth of a turn (`cut_projection`) and the perpendicular eigenplane that turns by two fifths (`phason_projection`), leaving the five-axis diagonal in the kernel of both.

Projected onto the cut, all five lattice axes have equal squared length two fifths and meet at multiples of a fifth of a turn.

In [ ]:
# Each mirror between consecutive axes swaps them; four swaps in sequence cycle all five axes.
neighbour_mirrors = (mirrors[:-1] - mirrors[1:]).normalized()                 # [4] Plane+
turn = neighbour_mirrors[0] * neighbour_mirrors[1] * neighbour_mirrors[2] * neighbour_mirrors[3]  # [] Rotor+
turning = turn >> Point                                                       # [] Point- <- Point-


def cycle_powers(step_map: Projection, n_powers: int) -> Iterator[Projection]:
    """Yields the first `n_powers` iterated compositions (`[] Point- <- Point-`) of `step_map`."""
    power = step_map                                                          # [] Point- <- Point-
    for _ in range(n_powers):
        yield power
        power = step_map(power)


def harmonic_projection(harmonic: int) -> Projection:
    """Cosine-weighted Fourier average over the five powers of `turning` projects onto the plane turned by `harmonic` fifths of a turn (`[] Point- <- Point-`)."""
    maps = stack(list(cycle_powers(turning, 5)))                              # [5] Point- <- Point-
    weights = 2 / 5 * np.cos(2 * np.pi * harmonic * np.arange(1, 6) / 5)
    return (maps * weights).sum(axis=0)                                       # [] Point- <- Point-


cut_projection = harmonic_projection(1)                                       # [] Point- <- Point-
phason_projection = harmonic_projection(2)                                    # [] Point- <- Point-
cut = cut_projection(axes[0]) & cut_projection(axes[1])                       # [] Area-
projected_axes = cut_projection(axes)                                         # [5] Point-
print("projected axis inner products:\n", (projected_axes[:, None] | projected_axes).to_array())

## Dual-cube acceptance and rhombus tiling

A square lattice face becomes a rhombus tile whenever the cut meets the unit three-cube dual to that face along the three complementary axes. Joining any two of those three axes with `cut` produces a plane that pairs to zero with both axes and with every displacement inside `cut`; dividing by the full five-volume spanned with the third axis gives the reciprocal reading frame by Cramer's rule without a metric. Pairing that frame with the offset from each face centre to the cut's anchor point tests whether all three dual-cube coordinates lie within half a unit.

In [ ]:
# Lattice half-span, generic zero-sum shift anchor in the five-axis space, and view half-width.
lattice_radius = scenarios.lattice_radius
origin_shift = (axes * np.array([0.12, 0.29, -0.11, 0.33, -0.63])).sum()       # [] Point-
view_extent = scenarios.view_extent

In [ ]:
def reciprocal_frames(cut: Area) -> Plane:
    """Cramer's rule in joins builds the three reading planes (`[10, 3] Plane+`) of each face's dual cube that vanish on `cut`."""
    others = axes[other_indices]                                              # [10, 3] Point-
    # Joining two complementary axes with the cut gives a plane blind to both and to the cut; dividing by the full five-volume normalizes its pairing on the remaining axis.
    cofactors = others[:, [1, 2, 0]] & others[:, [2, 0, 1]] & cut             # [10, 3] Plane+
    volumes = others[:, 0] & others[:, 1] & others[:, 2] & cut                # [10] Scalar+
    return cofactors / volumes[:, None]                                       # [10, 3] Plane+


def cube_margins(shift: Point, centres: Point, frames: Plane) -> Scalar:
    """Pairings of `frames` with the offset from `centres` to `shift` give the squared-half-width margins (`[..., 10, 3] Scalar+`) inside each dual cube."""
    offsets = frames & (shift - centres)[..., None]                           # [..., 10, 3] Scalar+
    return 1 / 4 - offsets.squared()                                          # [..., 10, 3] Scalar+


frames = reciprocal_frames(cut)                                               # [10, 3] Plane+

In [ ]:
def shadow_measures(cut_projection: Projection) -> tuple[Scalar, Scalar]:
    """Projected side joins and perpendicular dual-cube joins measure each face's tile area and selection frequency (`[10] Scalar+`, `[10] Scalar+`)."""
    sides = axes[side_indices]                                                # [10, 2] Point-
    others = axes[other_indices]                                              # [10, 3] Point-
    # Subtracting the cut projection leaves the complementary projection square to the cut.
    perpendicular_projection = Point - cut_projection                         # [] Point- <- Point-
    tiles = cut_projection(sides[:, 0]) & cut_projection(sides[:, 1])         # [10] Area-
    cube_shadows = (
        perpendicular_projection(others[:, 0])
        & perpendicular_projection(others[:, 1])
        & perpendicular_projection(others[:, 2])
    )                                                                         # [10] Antitrivector-
    return tiles.norm(), cube_shadows.norm()


def lattice_faces(cut_projection: Projection, lattice_radius: int) -> tuple[Point, Point]:
    """Integer lattice points within `lattice_radius` carry ten square face centres (`[n_points, 10] Point-`) and projected corner loops (`[n_points, 10, 4] Point-`)."""
    n_span = 2 * lattice_radius + 1
    n_points = n_span ** 5
    coordinates = np.array(list(itertools.product(range(-lattice_radius, lattice_radius + 1), repeat=5)))  # [n_points, 5]
    lattice_points = (axes * coordinates).sum(axis=-1)                        # [n_points] Point-
    sides = axes[side_indices]                                                # [10, 2] Point-
    centres = lattice_points[:, None] + sides.sum(axis=-1) / 2                # [n_points, 10] Point-
    # Displace half a unit along both face axes in cyclic order around each face centre.
    first_steps, second_steps = np.array([-1, 1, 1, -1]) / 2, np.array([-1, -1, 1, 1]) / 2
    corner_offsets = sides[:, None, 0] * first_steps + sides[:, None, 1] * second_steps  # [10, 4] Point-
    return centres, cut_projection(centres[..., None] + corner_offsets)


areas, volumes = shadow_measures(cut_projection)                              # [10] Scalar+, [10] Scalar+
centres, corners = lattice_faces(cut_projection, lattice_radius)              # [..., 10] Point-, [..., 10, 4] Point-

In [ ]:
def reading_planes(cut_projection: Projection) -> Plane:
    """Orthonormal directions in the cut dualize to two reading planes (`[2] Plane+`) aligned with the first projected axis."""
    right = cut_projection(axes[0]).normalized()                              # [] Point-
    # The commutator of two non-collinear directions in the cut generates rotations within the cut plane.
    cut_bivector = right.commutator(cut_projection(axes[1]))                  # [] Bivector+
    quarter_turn = (cut_bivector / cut_bivector.scalar_norm_squared().square_root() * (-np.pi / 4)).exp()  # [] Rotor+
    up = quarter_turn >> right                                                # [] Point-
    return stack([right, up]).dual()                                          # [2] Plane+


reading = reading_planes(cut_projection)                                      # [2] Plane+
initial_margins = cube_margins(origin_shift, centres, frames)                 # [..., 10, 3] Scalar+
save_figure(render.tiling_figure(reading, corners, areas, view_extent, initial_margins), "penrose_tiling");

Amber polygons show the thick rhombi and blue polygons show the thin rhombi, meeting edge to edge with five-fold rotational orientations across the patch without translational repetition.

Across the full lattice, face centres distribute uniformly over the three-dimensional space perpendicular to the cut, so each of the ten face orientations is selected in proportion to the perpendicular three-volume of its dual cube's shadow. Because an orthonormal lattice face and its dual three-cube split the same five-dimensional unit volume as the cut and its orthogonal complement, each dual cube's perpendicular shadow volume equals its projected tile area, making thick rhombi both larger and more frequent than thin ones by the golden ratio.

In [ ]:
golden = (1 + np.sqrt(5)) / 2
thick = areas > areas.mean()                                                  # [10]
selected = np.all(initial_margins > 0, axis=-1)                               # [..., 10]
print("tile areas:         ", areas.to_array())
print("dual-cube shadows:  ", volumes.to_array())
print("thick / thin area:  ", (areas[thick][0] / areas[~thick][0]).to_array(), "golden ratio:", golden)
print("tiles in the patch: ", selected[:, thick].sum(), "thick,", selected[:, ~thick].sum(), "thin")

## Phason flips

Displacing the cut along a direction inside `cut` leaves every reciprocal-frame pairing unchanged and merely slides the projected coordinates. Displacing it perpendicularly inside the two-dimensional `phason_projection` plane moves dual cubes across the acceptance boundary one at a time, so three rhombi sharing an interior vertex flip across their bounding hexagon while surrounding tiles remain stationary. Rendering the 135 frames around a closed phason circle takes about six seconds.

In [ ]:
# Perpendicular loop radius, frame count around the loop, and frame duration in milliseconds.
phason_radius = scenarios.phason_radius
n_frames = scenarios.n_frames
frame_duration_ms = scenarios.frame_duration_ms

In [ ]:
def phason_loop(phason_projection: Projection, origin_shift: Point, radius: float, n_frames: int) -> Point:
    """Rotors in the perpendicular phason plane sweep `origin_shift` around a closed circle of `n_frames` shifts (`[n_frames] Point-`)."""
    start = phason_projection(axes[0])                                        # [] Point-
    loop_bivector = start.commutator(phason_projection(axes[1]))              # [] Bivector+
    angles = np.linspace(0, 2 * np.pi, n_frames, endpoint=False)
    rotors = (loop_bivector / loop_bivector.scalar_norm_squared().square_root() * (angles / 2)).exp()  # [n_frames] Rotor+
    circle_points = rotors >> start                                           # [n_frames] Point-
    return origin_shift + radius * (circle_points - start) / start.norm()     # [n_frames] Point-


def margin_stream(shifts: Point, centres: Point, frames: Plane) -> Iterator[Scalar]:
    """Yields the dual-cube margins (`[..., 10, 3] Scalar+`) for each shift point along `shifts`."""
    for frame_shift in shifts:
        yield cube_margins(frame_shift, centres, frames)


loop_shifts = phason_loop(phason_projection, origin_shift, phason_radius, n_frames)  # [n_frames] Point-
streamed_margins = show_states(
    margin_stream(loop_shifts, centres, frames),
    render.phason_frames,
    reading,
    corners,
    areas,
    view_extent,
    name="penrose_phasons",
    duration_ms=frame_duration_ms,
    n_frames=n_frames,
);

Isolated hexagons of two thick and one thin rhombus, or two thin and one thick rhombus, snap between their two internal vertex positions as the perpendicular shift sweeps around its circle, returning every tile to its initial state when the loop closes.

In [ ]:
# checks
# Five applications of the axis-cycling turn return every axis to itself.
fifth_power = list(cycle_powers(turning, 5))[-1]                              # [] Point- <- Point-
assert ((fifth_power(axes) - axes).norm() < 1e-12).all()
# Each face's reciprocal frame pairs as the identity on its three dual-cube axes and vanishes on the cut.
others = axes[other_indices]                                                  # [10, 3] Point-
frame_pairings = frames[:, None] & others[..., None]                          # [10, 3, 3] Scalar+
assert ((frame_pairings - np.eye(3)).abs() < 1e-12).all()
assert ((frames & projected_axes[:, None, None]).abs() < 1e-12).all()
# Dual-cube shadow volumes equal projected tile areas, and thick-to-thin areas and patch counts follow the golden ratio.
assert ((volumes - areas).abs() < 1e-12).all()
assert ((areas[thick] / areas[~thick][:, None] - golden).abs() < 1e-12).all()
assert abs(selected[:, thick].sum() / selected[:, ~thick].sum() - golden) < 5e-2 * golden
assert len(streamed_margins) == n_frames
assert ((streamed_margins[0] - initial_margins).abs() < 1e-12).all()